# 25 · Autoencodeurs et VAE — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch25_autoencodeurs/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 25.Q1 à 25.Q12, 25.R1 à 25.R3, 25.1 à 25.4, 25.6, 25.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 25.14 à 25.23 | des autoencodeurs denses : une seule image, MNIST, l'espace latent, l'interpolation, la taille du goulot, l'autoencodeur linéaire face à l'ACP (*prochaine session de génération*) | | | |
| B | 25.24 à 25.27 | des autoencodeurs convolutifs et débruiteurs : 147 latents, un 🐛 de débruiteur, sur-échantillonnage ou convolutions à pas (*prochaine session de génération*) | | | |
| C | 25.28 à 25.39 | le VAE : ta librairie `vae.py`, un VAE PyTorch, générer, β, les anomalies, un VAE en NumPy, le défi de débruitage (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Construire et entraîner en PyTorch des autoencodeurs denses, convolutifs et débruiteurs, et explorer leur espace latent (carte 2D, bruit, interpolation).
- Relier l'autoencodeur linéaire à l'ACP ; mesurer un débruitage (PSNR).
- Écrire en NumPy le cœur du VAE (`mylearn.vae`), entraîner un VAE, régler β, générer, détecter des anomalies.

**Rappel express.** $\mathbf{z} = f_\theta(\mathbf{x})$, $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$, la cible est l'entrée. Convolution transposée : $H_{out} = (H_{in} - 1)s - 2p + k + p_{out}$. PSNR $= 10 \log_{10}(1/\text{MSE})$ pour des pixels dans $[0, 1]$. VAE : $\mathbf{z} = \boldsymbol{\mu} + e^{\frac{1}{2}\log\boldsymbol{\sigma}^2} \odot \boldsymbol{\varepsilon}$, loss = reconstruction + $\beta$ KL, $\mathrm{KL} = \tfrac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 25.1 à 25.4 et 25.6, ∂ 25.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`) ; une forme ou un vecteur : entre guillemets (`"(3, 5, 1)"`, `"(1, -2)"`) ; un mot : entre guillemets (`"ABCD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 25.5 et 25.8 et la démarche de ∂ 25.7, les estimations 🧮 25.9 et 25.10, la réflexion (🗣️ 25.11, ⚖️ 25.12, 📄 25.13) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math


def dense(n_in, n_out):
    """Parameters of nn.Linear(n_in, n_out): weights and biases."""
    return (n_in + 1) * n_out


def conv_params(c_in, c_out, k):
    """Parameters of nn.Conv2d(c_in, c_out, k) (or ConvTranspose2d): weights and one bias per output channel."""
    return c_out * (c_in * k * k + 1)


def conv_out(h, k, s=1, p=0):
    """Height after nn.Conv2d(kernel_size=k, stride=s, padding=p)."""
    return (h + 2 * p - k) // s + 1


def tconv_out(h, k, s=1, p=0, op=0):
    """Height after nn.ConvTranspose2d(kernel_size=k, stride=s, padding=p, output_padding=op)."""
    return (h - 1) * s - 2 * p + k + op


def mlp_params(sizes):
    return sum(dense(a, b) for a, b in zip(sizes, sizes[1:]))


def gaussian_kl(mu, var):
    """KL(N(mu, var) || N(0, 1)) in nats."""
    return 0.5 * (mu ** 2 + var - 1 - math.log(var))


def kl_bits(p, q):
    return sum(pi * math.log2(pi / qi) for pi, qi in zip(p, q))


def letter_mix(word_a, word_b):
    """The book's letter blend: the middle letter (the earlier one when there are two)."""
    out = ""
    for a, b in zip(word_a, word_b):
        lo, hi = sorted((ord(a), ord(b)))
        out += chr((lo + hi) // 2)
    return out


ENC_DEEP = [784, 512, 256, 128, 20]
P_ENC_DEEP = mlp_params(ENC_DEEP)
P_DEC_DEEP = mlp_params(ENC_DEEP[::-1])
P_ENC_CONV = conv_params(1, 16, 3) + conv_params(16, 8, 3) + conv_params(8, 3, 3)
P_DEC_CONV = conv_params(3, 16, 3) + conv_params(16, 1, 3)
A_25_2 = dict(cx=2.0, cy=2.0, r=1.0, g=1.0)
B_25_2 = dict(cx=10.0, cy=6.0, r=4.0, g=0.4)
MIX_25_2 = {key: 0.75 * A_25_2[key] + 0.25 * B_25_2[key] for key in A_25_2}

**25.Q1 — Avec ou sans perte : le test de l'inversibilité**

In [3]:
wb.record("25.Q1a", "AC", mistakes={"arrondir au degré : peut-on retrouver les décimales ?": "ABC",
          "JPEG jette de l'information, même en qualité 90": "ACE",
          "une photo en niveaux de gris : peut-on retrouver ses couleurs ?": "ACF",
          "une lettre sur deux : peut-on retrouver les lettres retirées ?": "ACD",
          "écrire en binaire change la forme d'un nombre : peut-on revenir au décimal ?": "C",
          "un fichier ZIP : retrouve-t-on le fichier exact en le décompressant ?": "A"})
wb.record("25.Q1b", True, mistakes={"applique le test de l'inversibilité : existe-t-il une autre façon d'être sans perte ?": False})
wb.record("25.Q1c", "B", mistakes={"l'erreur de reconstruction est petite mais pas nulle : retrouve-t-on l'image exacte ?": "A"})
wb.record("25.Q1d", 2_400_000 / 300_000, decimals=3, mistakes={"on divise la taille d'origine par la taille compressée, pas l'inverse": 0.125,
          "c'est la part des octets supprimés, pas le facteur de compression": 0.875})
wb.record("25.Q1e", False, mistakes={"compte les fichiers courts et les fichiers longs : y a-t-il assez de fichiers courts pour tous ?": True})

WB_ANSWER {"hash": "a863b7735631829bd1122a2bfb63862fe96e54f7971ae0f15447f74fbae6b68c", "id": "25.Q1a", "kind": "str", "mistakes": {"4b6edfa3f2ccfe3a18f2c8748cabcd90d093b3ffe15dc2f47709e3efaa898b95": "arrondir au degré : peut-on retrouver les décimales ?", "a5fcabb0ccf5b4bb02d17a17d8db26bdeb679267a4f264c128a644d740827efb": "JPEG jette de l'information, même en qualité 90", "bd35941dd9eb331c3f536a3148aff05211ce6a5e74c04ced4c8ea161f5907eae": "écrire en binaire change la forme d'un nombre : peut-on revenir au décimal ?", "c905d8b4481b57d57dfd4155c7750173dad900861ee85d19036377f4b65ce426": "un fichier ZIP : retrouve-t-on le fichier exact en le décompressant ?", "cfa914c17ea75b2c66202484332256377cc74b07469de72df87114dc03334c30": "une lettre sur deux : peut-on retrouver les lettres retirées ?", "dac0addf8caf292b64fba12274af86e63dc5576394bda7c2766f66be5043e462": "une photo en niveaux de gris : peut-on retrouver ses couleurs ?"}}
📝 Ex 25.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "99c77

**25.Q2 — Pourquoi JPEG et MP3 battent un autoencodeur généraliste**

In [4]:
wb.record("25.Q2a", "C", mistakes={"JPEG n'apprend rien : ses règles ont été écrites par des ingénieurs": "A",
          "JPEG jette de l'information : relis le §25.2.2": "B",
          "JPEG ne garde aucun catalogue d'images": "D"})
wb.record("25.Q2b", "C", mistakes={"le code a été appris sur un seul type d'images": "A",
          "simple pour toi, pas pour un code appris sur d'autres images": "B"})
wb.record("25.Q2c", False, mistakes={"relis la première page du chapitre : qui aurait quelque chose à craindre ?": True})
wb.record("25.Q2d", "AC", mistakes={"il manque un des deux usages : relis le §25.1": "A",
          "un seul usage : le livre en donne deux, relis le §25.1": "C",
          "chiffrer n'est pas comprimer : un autoencodeur ne cache rien": "ABC",
          "dater une image n'est pas un usage d'autoencodeur": "ACD"})
wb.record("25.Q2e", 128_000 * 180 / 8 / 1e6, decimals=2, mistakes={"8 bits par octet : as-tu divisé par 8 ?": 23.04,
          "3 minutes, c'est 180 secondes": 0.048})

WB_ANSWER {"hash": "5cfb442fec4855329ed06bdc16040039eb1227568849e7f6f1489efad6d81ac0", "id": "25.Q2a", "kind": "str", "mistakes": {"5026043125b0b9d37aa340149d5f1d3fcf61a0ded7fe8132df9e6c954da566c5": "JPEG ne garde aucun catalogue d'images", "81512e0d8153a19ab6a74de5eeec2a448c72e102fd5cebeb1dfd08db0629dbed": "JPEG n'apprend rien : ses règles ont été écrites par des ingénieurs", "e6d9b99e4fa60a941a147b6ccf79258410dce27b5b564fae5b4f35d54e11da90": "JPEG jette de l'information : relis le §25.2.2"}}
📝 Ex 25.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "09606bd5f7b4e6217e24fc3ca4b046fe0423cf715526ba3491925fa4bfc0a4a1", "id": "25.Q2b", "kind": "str", "mistakes": {"03b119b1461975186ed5f1f4cc4e5b0f835d8710332d1f949afca38edb135082": "le code a été appris sur un seul type d'images", "fbfbc4a232f8da4d122e3d70af9000b37314d07a633774d92a2d80b911e9a12f": "simple pour toi, pas pour un code appris sur d'autres images"}}
📝 Ex 25.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "e9c4528d2e2beeada

**25.Q3 — Mélange de contenu ou mélange de paramètres ?**

In [5]:
wb.record("25.Q3a", "A", mistakes={"on mélange les pixels, pas ce qui décrit les objets": "B"})
wb.record("25.Q3b", letter_mix("LUNE", "SOIR"), mistakes={"quand il y a deux lettres au milieu, la règle prend la plus proche du début de l'alphabet": "PRLL",
          "la lettre du milieu, pas la première lettre de chaque paire": "LOIE"})
wb.record("25.Q3c", False, mistakes={"relis ce que donne le mélange des deux noms de fruits du livre": True})
wb.record("25.Q3d", "C", mistakes={"un code court peut être mal rangé : que faut-il entre des objets voisins ?": "A",
          "binaire ou non, ce qui compte est la place des codes les uns par rapport aux autres": "B",
          "« sans perte » ne dit rien de la place des codes entre eux": "D"})

WB_ANSWER {"hash": "b29201de1848cec878554cd9e9503d2450f7cfdeb54daac9345353cadaec854e", "id": "25.Q3a", "kind": "str", "mistakes": {"d51ee2f827576c7c89a28ed979a7d182724463f2c524a3ee8b527114c05c668b": "on mélange les pixels, pas ce qui décrit les objets"}}
📝 Ex 25.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "18c030fbb9c97149d1a3d9a493c6f8b9f09c0b65a3cdaa5a7cc59de9e7a41f2f", "id": "25.Q3b", "kind": "str", "mistakes": {"6081a5ec45b0ad2ca259e0a562d380b130c50e53948bbd775b462e48405e065c": "la lettre du milieu, pas la première lettre de chaque paire", "e252dad3d22351239648fd351186453c5d78e06d2a177ee21a4aaeecc0b52154": "quand il y a deux lettres au milieu, la règle prend la plus proche du début de l'alphabet"}}
📝 Ex 25.Q3b : réponse enregistrée (str).
WB_ANSWER {"hash": "46129e4e3a4a9ded624e7bb6df208f3cfe591d3c62f885cd9076b8be49cd36db", "id": "25.Q3c", "kind": "bool", "mistakes": {"6fca7e2541852f86c35a598cc279c138adc05c9e7c106763eea0ba6bbea90ecc": "relis ce que donne le mélange des deux

**25.Q4 — Goulot, code, variables latentes : vrai ou faux**

In [6]:
wb.record("25.Q4a", False, mistakes={"la cible d'un autoencodeur, d'où vient-elle ?": True})
wb.record("25.Q4b", True, mistakes={"que cherche à reproduire un autoencodeur ?": False})
wb.record("25.Q4c", True, mistakes={"où se trouve le goulot, et que contient-il ?": False})
wb.record("25.Q4d", "C", mistakes={"il n'y a aucun label à fournir": "A",
          "c'est le terme du livre, corrigé dans la fiche": "B",
          "il n'y a ni agent ni récompense": "D"})
wb.record("25.Q4e", True, mistakes={"si le code est aussi grand que l'entrée, que peut faire le réseau le plus simple ?": False})

WB_ANSWER {"hash": "df52a432711f319c7f72e3e6fad0a66fa97fc2ddb59233784df8419e7969e893", "id": "25.Q4a", "kind": "bool", "mistakes": {"3e9c9fb6baead5a1d79080fc2b35efd361b21fb78aeaa42832086a453df4a35d": "la cible d'un autoencodeur, d'où vient-elle ?"}}
📝 Ex 25.Q4a : réponse enregistrée (bool).
WB_ANSWER {"hash": "de474429aa220d76d6c32326ab1eaea7085ec8276af638167ba64b005f47c02c", "id": "25.Q4b", "kind": "bool", "mistakes": {"db3203584abc3d93d10914278254addd161b41cd4c0168cf1d6ac9339c5d1b85": "que cherche à reproduire un autoencodeur ?"}}
📝 Ex 25.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "cb345cd4706162d180e619a945a58146a9b2bdefa3b329d2c925564820096216", "id": "25.Q4c", "kind": "bool", "mistakes": {"3b10992ca0befe434d0df4fb23c47ffc3c54df344cf02b3b1f9fc3dc95728280": "où se trouve le goulot, et que contient-il ?"}}
📝 Ex 25.Q4c : réponse enregistrée (bool).
WB_ANSWER {"hash": "946c5b8be96dc1e74fbe8a9dc099d847d3453c0f9d62ea46ce53004f3c8ec367", "id": "25.Q4d", "kind": "str", "mistakes"

**25.Q5 — Le réseau qui ne savait redessiner qu'une image**

In [7]:
wb.record("25.Q5a", 2 * 4096 * 10, mistakes={"deux couches : l'encodeur et le décodeur ont chacune leurs poids": 4096 * 10,
          "une image de 64 × 64 a 64 × 64 pixels, pas 64 + 64": 2 * 128 * 10,
          "relis la question : sans les biais": 2 * 4096 * 10 + 4096 + 10})
wb.record("25.Q5b", 64 * 64, mistakes={"un biais par neurone de la couche de sortie, qui a un neurone par pixel": 10})
wb.record("25.Q5c", "C", mistakes={"les biais de la sortie ne dépendent pas de l'entrée": "A",
          "rien n'est tiré au hasard dans un réseau entraîné": "B",
          "le réseau accepte toute image de la bonne taille": "D"})
wb.record("25.Q5d", False, mistakes={"relis l'histoire du tigre : que montre l'image noire ?": True})
wb.record("25.Q5e", "C", mistakes={"sur ses images d'entraînement, un réseau qui mémorise semble excellent": "A",
          "le nombre d'epochs ne dit pas ce que le réseau a appris": "B",
          "la taille du goulot ne dit pas si le réseau triche": "D"})

WB_ANSWER {"hash": "75b19dfb3bb27e592eb9d7bc94caaa563004ad0c2faa5c330af03fad9302cfce", "id": "25.Q5a", "kind": "int", "magnitude": 4, "mistakes": {"5ab54d40751fa679bb7865e39a511f4b8dcdcb8c46e33076674e4fce3e0035ff": "deux couches : l'encodeur et le décodeur ont chacune leurs poids", "6c3074f5956cc579a0b31d3612c8bba79758184ae2b98e7c69223160c09c4a68": "relis la question : sans les biais", "937319c2e46792ec1030ed0f317099d923c1cfd29338e9c59c143a72e2f73a2f": "une image de 64 × 64 a 64 × 64 pixels, pas 64 + 64"}, "sign": 1}
📝 Ex 25.Q5a : réponse enregistrée (int).
WB_ANSWER {"hash": "19469699401dfbaff5d534dfeaebce82884150b4faa1a071df60c723cac4861e", "id": "25.Q5b", "kind": "int", "magnitude": 3, "mistakes": {"28675a29bf15811c2878abd7460f972f84b69885d02c2ad9fdbe525a863e52f5": "un biais par neurone de la couche de sortie, qui a un neurone par pixel"}, "sign": 1}
📝 Ex 25.Q5b : réponse enregistrée (int).
WB_ANSWER {"hash": "800792e6706b84bf8bb472476d4695261cfe741e0741c8e28253002ffc3a0920", "id": 

**25.Q6 — Profondeur ou nombre de latents : qui améliore quoi ?**

In [8]:
wb.record("25.Q6a", "C", mistakes={"moins de latents, c'est moins d'information dans le code": "A",
          "relis les figures 25.18 à 25.20": "B"})
wb.record("25.Q6b", "B", mistakes={"plus de couches, c'est plus de capacité pour calculer et décoder le code": "A",
          "relis la figure 25.22": "C"})
wb.record("25.Q6c", "AC", mistakes={"il manque une des deux conditions du livre": "A",
          "une seule condition : le livre en pose deux": "C",
          "un autoencodeur n'a pas de labels": "ABC",
          "avec un learning rate nul, rien n'est appris": "ACD"})
wb.record("25.Q6d", False, mistakes={"le livre parle d'une habitude, pas d'une obligation": True})
wb.record("25.Q6e", "B", mistakes={"2 latents donnent des reconstructions très floues": "A",
          "la loss reste celle de la reconstruction": "C",
          "le décodeur sert toujours": "D"})

WB_ANSWER {"hash": "8540cf30729880fba28cf09a9fda90e68698d682972cc96f4db7a9294ccf97b1", "id": "25.Q6a", "kind": "str", "mistakes": {"0f5f92f29f11a71116605e8ace6b6fb8d3284aad88c1dc00aa4f6bd840ecb5f3": "relis les figures 25.18 à 25.20", "d50f049d168686f95b32bfbeaca3da0282f314a2b0a41fadcdd4687fca3a704c": "moins de latents, c'est moins d'information dans le code"}}
📝 Ex 25.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "bd0ef4bf8e7d4cda828d935c09cb2fedd837a87b2e41a05e3c28be83f7bdecc4", "id": "25.Q6b", "kind": "str", "mistakes": {"0b7644ea9f41d664b32ceda62e9eefa2fc9bce123fcd48c12aac24f6b7b5052f": "plus de couches, c'est plus de capacité pour calculer et décoder le code", "435e26cbc0f727f36499eb59da7314e6345e5f2e5b3220131b25bf35a9c6e6a5": "relis la figure 25.22"}}
📝 Ex 25.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "bbee0cc53d7aec8accf215bebe1fbb11109773646834bf653587e6263070ce56", "id": "25.Q6c", "kind": "str", "mistakes": {"245843ed74cffda130f6a120798ee9ffc4959ff5f8df21b2435553

**25.Q7 — Du bruit dans l'espace latent d'un autoencodeur classique**

In [9]:
wb.record("25.Q7a", "B", mistakes={"compare la taille du bruit à celle des latents": "A"})
wb.record("25.Q7b", "A", mistakes={"compare la taille de ce bruit à celle des latents": "B"})
wb.record("25.Q7c", "A", mistakes={"chaque chiffre dépend plus ou moins de chaque latent": "B"})
wb.record("25.Q7d", False, mistakes={"relis ce que donnent des latents tirés au hasard": True})
wb.record("25.Q7e", "A", mistakes={"relis la figure 25.28": "B"})

WB_ANSWER {"hash": "c1affbfa48e90d3f561d243542eae5b7f8e96e2ad1e384b72c511d883a3a3b32", "id": "25.Q7a", "kind": "str", "mistakes": {"bfaee8f0aad6d79f79d51ee71465dcc9fa53b131a07745d3b5a1d3ccc3b3d360": "compare la taille du bruit à celle des latents"}}
📝 Ex 25.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "8bc0795dff132202501dcb05ed134f3fc4de1995ae02881cce1258fd2c87573d", "id": "25.Q7b", "kind": "str", "mistakes": {"6fe16ef5148ac7d8717fee4b9b83fca8a0c7fd10720615293b4e59bc3d7729ec": "compare la taille de ce bruit à celle des latents"}}
📝 Ex 25.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "e17e08f0c1f2110e33fa7ff75a4806d154f1bb8ce75f6022f470e30572603d07", "id": "25.Q7c", "kind": "str", "mistakes": {"753ac9485dc7e5e949e49f0f7fd21a5f118554c41e3c15d2322194ae66b114bf": "chaque chiffre dépend plus ou moins de chaque latent"}}
📝 Ex 25.Q7c : réponse enregistrée (str).
WB_ANSWER {"hash": "21818857e30de78d9eeb6cf0b44a466e11bfde221b88e5728f08f0cb01abdd43", "id": "25.Q7d", "kind": "bool",

**25.Q8 — Lire une carte latente à deux dimensions**

In [10]:
wb.record("25.Q8a", "A", mistakes={"la carte ne dit rien de la fréquence des chiffres": "B",
          "les points sont placés par l'encodeur, pas par le dessin": "C",
          "des codes voisins pour des chiffres différents, est-ce bon signe ?": "D"})
wb.record("25.Q8b", "B", mistakes={"aucun exemple n'a de code dans cette zone": "A",
          "le décodeur rend toujours une image, pas forcément noire": "C"})
wb.record("25.Q8c", "B", mistakes={"on mélange les codes, pas les pixels": "A"})
wb.record("25.Q8d", False, mistakes={"relis le §25.5.3 : que donnent certains mélanges ?": True})
wb.record("25.Q8e", "(5, 3)", mistakes={"le milieu est la moyenne des deux points : as-tu divisé par 2 ?": "(10, 6)",
          "c'est la différence des deux codes, pas leur milieu": "(6, -6)",
          "moyenne coordonnée par coordonnée : la première avec la première": "(4, 4)"})

WB_ANSWER {"hash": "87c99afe2e2f3f9ee819d87a6689c6f49773164a617e6fc279a17ea09fe8ce1a", "id": "25.Q8a", "kind": "str", "mistakes": {"5baaad89c15947ee1abce8fa3f3467faa07ab4d4bcce0b741a5a8a13ec596a6f": "la carte ne dit rien de la fréquence des chiffres", "63ddef8ac48de52a200dbaa6ebbd979fe097cac8b3c1be9544ced2d82a5dcd27": "les points sont placés par l'encodeur, pas par le dessin", "e9ebc45c6176537acba561b072cf868defbe4ff01427b8f65c7d613fccfb980e": "des codes voisins pour des chiffres différents, est-ce bon signe ?"}}
📝 Ex 25.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "33c3f7a32bb6c25499053fd170f76aa23b26bb2a54a4c00b4161c8f528c31c7c", "id": "25.Q8b", "kind": "str", "mistakes": {"19f6ee96bd36127dc6e05546792c732a46a616727526adbd9b5feacf64245d62": "le décodeur rend toujours une image, pas forcément noire", "558055d76b6fc3585695c92704cf0fc74a3389fe854cf55319125dfa754dbe9d": "aucun exemple n'a de code dans cette zone"}}
📝 Ex 25.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "d9b564

**25.Q9 — Autoencodeur linéaire et ACP**

In [11]:
wb.record("25.Q9a", "B", mistakes={"rien ne force les latents à être orthogonaux ni rangés par variance": "A",
          "pas n'importe lequel : celui qui minimise l'erreur de reconstruction": "C",
          "garde-t-on les directions de plus grande ou de plus petite variance ?": "D"})
wb.record("25.Q9b", False, mistakes={"relis l'encadré « Le livre, corrigé » du §25.6": True})
wb.record("25.Q9c", True, mistakes={"l'ACP commence par centrer les données : qui le fait dans l'autoencodeur ?": False})
wb.record("25.Q9d", "A", mistakes={"aucun modèle ne garantit une erreur nulle": "B",
          "un autoencodeur apprend sur des données": "C",
          "des axes orthogonaux classés par variance, c'est ce que fournit l'ACP": "D"})
wb.record("25.Q9e", 0.6 * 5 + 0.8 * 0, decimals=1, mistakes={"le produit scalaire associe chaque coordonnée de u à la même coordonnée de x": 4.0,
          "c'est la norme de x, pas sa coordonnée sur u": 5.0,
          "le code est un seul nombre, la coordonnée sur la droite ; 1,8 est une coordonnée de la projection": 1.8})

WB_ANSWER {"hash": "c4d16a15b993d3fb9e4860dcf17644d605008d46f8229ab950a95728eb1cc536", "id": "25.Q9a", "kind": "str", "mistakes": {"98e28619298275ec417c103cc3cccc7da4d1849b07950b3e10b072103fc28e79": "pas n'importe lequel : celui qui minimise l'erreur de reconstruction", "d22cb6bc0c09952af5b29bc03209ef3fcda870756658aac77be2bf735c39ef4e": "rien ne force les latents à être orthogonaux ni rangés par variance", "fd2824fe0357b216b18bb6e8cfb3cd990a7b19067a26ddfab1c59b7b95931496": "garde-t-on les directions de plus grande ou de plus petite variance ?"}}
📝 Ex 25.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "3cc94dec977edad55d1000368ca8e88a4c7a269041d2818cfd1d2263c8210eb3", "id": "25.Q9b", "kind": "bool", "mistakes": {"fd43e73ee0839c857721ccd1a6be8a787d84b405711d0c4faafa613554a8834d": "relis l'encadré « Le livre, corrigé » du §25.6"}}
📝 Ex 25.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "e8d92a7def9f4f91fe7e2b3151148aaa3989df70dcfff6dece68e1b9163e7cae", "id": "25.Q9c", "kind": "bo

**25.Q10 — Autoencodeurs convolutifs et débruiteurs**

In [12]:
wb.record("25.Q10a", "B", mistakes={"le réseau doit apprendre à rendre l'image propre": "A",
          "si la cible est l'entrée bruitée, que va apprendre le réseau ?": "C",
          "sans bruit à l'entrée, il n'y a rien à débruiter": "D"})
wb.record("25.Q10b", False, mistakes={"relis ce que le livre dit du code de son débruiteur": True})
wb.record("25.Q10c", "A", mistakes={"écrêter ne change pas la durée de l'entraînement": "B",
          "écrêter ne retire pas le bruit": "C",
          "la loss ne devient pas nulle": "D"})
wb.record("25.Q10d", "B", mistakes={"des couches denses perdent la structure de l'image": "A",
          "le dropout ne change pas la taille d'une image": "C",
          "la batchnorm ne change pas la taille d'une image": "D"})
wb.record("25.Q10e", False, mistakes={"le réseau a-t-il vu des photos pendant son entraînement ?": True})

WB_ANSWER {"hash": "3d1291a959048a6de2da57ed9eb76779f4fdaa3226a2be7d9ec9c03eb2058b7e", "id": "25.Q10a", "kind": "str", "mistakes": {"2b8139ad61c308ec05791429d6fb79602574d27e6854ed9e522d85fcf0d62d28": "si la cible est l'entrée bruitée, que va apprendre le réseau ?", "88cbcb253af9eeb18ed8b55e81bbda172bc106243d91bbd8cbe4bcf0f0789ee0": "sans bruit à l'entrée, il n'y a rien à débruiter", "bc6924e27ff074f82d9e29e7c2ba25781ab7e6a9fa9c0483881254e1b59a771b": "le réseau doit apprendre à rendre l'image propre"}}
📝 Ex 25.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "5485a3ef780c2dd893a5443a26feb68bfd9f6e1581b8aaf8e019d2abecf1a27c", "id": "25.Q10b", "kind": "bool", "mistakes": {"1363ea7803f173229bcfe969311ecaf7f3ec3946c23e57006b64e953f0c38851": "relis ce que le livre dit du code de son débruiteur"}}
📝 Ex 25.Q10b : réponse enregistrée (bool).
WB_ANSWER {"hash": "72d901a844be10eb219278abb8cb0ddbfcdb7722c3e78bcfa028f16a8ce8cc04", "id": "25.Q10c", "kind": "str", "mistakes": {"05e93330a6a26fa2b3

**25.Q11 — Une entrée jamais vue : un tigre chez les chiffres**

In [13]:
wb.record("25.Q11a", "C", mistakes={"le réseau ne sait reconstruire que des chiffres": "A",
          "le réseau rend toujours quelque chose de visible": "B",
          "le résultat dépend de l'entrée": "D"})
wb.record("25.Q11b", True, mistakes={"un code appris sur des chiffres représente-t-il bien autre chose ?": False})
wb.record("25.Q11c", False, mistakes={"une image presque vide est-elle difficile à reconstruire ?": True})
wb.record("25.Q11d", "A", mistakes={"la photo n'avait pas de cadre": "B",
          "le padding ne colore pas la sortie du décodeur": "C",
          "c'est systématique, pas un hasard": "D"})
wb.record("25.Q11e", "BC", mistakes={"une des réponses manque": "B",
          "il manque une réponse": "C",
          "un chiffre du test MNIST vient de la même distribution que l'entraînement": "ABC"})

WB_ANSWER {"hash": "41bd00cbda57cdfb4da1d6ee3ab7ad7774d70aaebf5f529a0ba04cf72c1e85bf", "id": "25.Q11a", "kind": "str", "mistakes": {"5981008e85ac350197451d18dfb84571669f61d8669b0335a56c1862a186ac3d": "le résultat dépend de l'entrée", "7c7fc447f9c906c9b820b03931b482a18a3fce6c4e9d453c652150f000f9f4e5": "le réseau ne sait reconstruire que des chiffres", "a76b4076f5af1e51faa39d419b781ff364287bc578576707eced15ce93067307": "le réseau rend toujours quelque chose de visible"}}
📝 Ex 25.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "4cd8beb354bc374a3ea6e4be1dff533771387a0740590d3e429a1ce4e8cf2856", "id": "25.Q11b", "kind": "bool", "mistakes": {"5907c8a4906ff97694994b4a0304987ecc2050be2ed456cf0345cfe00c2818b7": "un code appris sur des chiffres représente-t-il bien autre chose ?"}}
📝 Ex 25.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "ed621bb6cf756ef7a4bb6387b4fe2522289d474708a7170dd6a1544e2b3927b1", "id": "25.Q11c", "kind": "bool", "mistakes": {"45fd11b07c7f7c9631fe8320afd6e13bc25

**25.Q12 — Les trois exigences d'un VAE et l'astuce de reparamétrisation**

In [14]:
wb.record("25.Q12a", "ABC", mistakes={"des latents entiers n'aident pas à tirer des codes": "ABCD",
          "le livre veut des codes proches pour des entrées proches, pas un code par classe": "ABCE",
          "il manque une exigence : relis le §25.9.2": "AB",
          "il manque une des trois exigences": "BC",
          "il manque une exigence": "AC"})
wb.record("25.Q12b", "B", mistakes={"un VAE ne rend pas un seul nombre par latent": "A",
          "il n'y a pas de label": "C",
          "relis la description de l'encodeur au §25.9.2": "D"})
wb.record("25.Q12c", False, mistakes={"tiré tel quel, z est-il une fonction de μ et de σ ?": True})
wb.record("25.Q12d", 1.5 + 0.5 * (-2), decimals=2, mistakes={"σ multiplie ε : z = μ + σ ε": -0.5,
          "ε vaut −2, pas 2": 2.5,
          "σ = 0,5 est l'écart-type : ne l'élève pas au carré": 1.0})
wb.record("25.Q12e", False, mistakes={"le tirage de ε change-t-il d'un passage à l'autre ?": True})

WB_ANSWER {"hash": "e42f28678b03b4a3b511e40676c0ff0aec50dc64af421b0f9d1d6b2c4dbe2fbc", "id": "25.Q12a", "kind": "str", "mistakes": {"837892ba095191bd0e34483db61e420df7ab7035984bf8e4789cf0662a131d6c": "des latents entiers n'aident pas à tirer des codes", "a30d6646b67b7b8afc6c84a2966df906cac8c1b477aab54ff3af4fc4f35a0064": "il manque une exigence", "a4784646afc6174873b3f9c35360ce9b167dfc8484e73b2dfa64516947927fb0": "il manque une des trois exigences", "b410644eb96d4629c2b52d5daf788ff3b07f1ea7fc27cf0c3db45b89004529a1": "il manque une exigence : relis le §25.9.2", "dcf1ded13db3fba8b31d5d226576c7ccb43ec767133ed5c0ca28157aaa5d49da": "le livre veut des codes proches pour des entrées proches, pas un code par classe"}}
📝 Ex 25.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "c9169340311cefa14e4ca971297c4169070cc0a640af6e280557578d1dae1fe4", "id": "25.Q12b", "kind": "str", "mistakes": {"43b4152b531c64f329c91b1c798e05d0c68007c9d3ad69206b86b0fb4d5e7df3": "un VAE ne rend pas un seul nombre par 

**25.R1 — Quelles augmentations garder pour des chiffres manuscrits ?**

In [15]:
wb.record("25.R1a", "ABE", mistakes={"un chiffre retourné gauche-droite reste-t-il le même chiffre ?": "ABCE",
          "un 6 tourné de 180°, est-ce encore un 6 ?": "ABDE",
          "il manque une transformation plausible": "AB",
          "il en manque une": "AE",
          "une transformation plausible manque": "BE"})
wb.record("25.R1b", False, mistakes={"le test doit représenter les vraies données : faut-il le transformer ?": True})
wb.record("25.R1c", True, mistakes={"chaque epoch voit-elle les mêmes images bruitées ?": False})

WB_ANSWER {"hash": "96f188f53be47f7385f3ae6eef53748a74153960ce991a6636e9f63fdf61d044", "id": "25.R1a", "kind": "str", "mistakes": {"1d225d287f74c68a3c88c3f6b9b84290bad1bcd0f05a9d736599134d406446ae": "une transformation plausible manque", "5bb17d55b7d231119b52ec9b4801a5263547eef8026da55b37c41764579acad7": "il en manque une", "5d7f583bbabf352f6e2deda8af943a2b1d72c43cd9f75e6d5c7dcb43f5a8a6b4": "un 6 tourné de 180°, est-ce encore un 6 ?", "ca534d6855b000dac974db56dd6925b34a8d4f8b4f60fd014f28d9354a452d14": "il manque une transformation plausible", "d80af25940780e6aef42eb64a6004ad64aeea068d9ddfa7a57c3b6b41aa9c6b1": "un chiffre retourné gauche-droite reste-t-il le même chiffre ?"}}
📝 Ex 25.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "783a3f702847ce81f7ddcd3c925e124ac9f61f9cb931e1b33f244b419dbffe28", "id": "25.R1b", "kind": "bool", "mistakes": {"f37008a2c00af694d41c1bbcaea1b37b19a2e7170772f1041e9cb288a9a24205": "le test doit représenter les vraies données : faut-il le transformer ?"}}


**25.R2 — Résumer une séquence entière dans l'état final d'un LSTM**

In [16]:
wb.record("25.R2a", "(1, 16, 32)", mistakes={"h_n garde l'axe des couches en premier, même avec batch_first=True": "(16, 1, 32)",
          "h_n ne contient que le dernier pas de temps : c'est output qui les a tous": "(16, 50, 32)",
          "h_n a trois axes : couches, exemples, état": "(16, 32)"})
wb.record("25.R2b", 50 * 8 / 32, decimals=1, mistakes={"compte toutes les valeurs de la séquence : 50 pas de 8 features": 1.6,
          "le code a 32 valeurs, pas 8": 50.0})
wb.record("25.R2c", True, mistakes={"un encodeur et un décodeur peuvent être des réseaux récurrents": False})

WB_ANSWER {"hash": "c88962c0ac36ddbc79b89313ccd326e5ef3ceb72789e3e37dc381ba6ae40777c", "id": "25.R2a", "kind": "str", "mistakes": {"00d432b9181336594d5ee181ef8ac4778730910e2a2cca779d75531d560fa204": "h_n garde l'axe des couches en premier, même avec batch_first=True", "463c5fa1332cecc01f9384a019de15289f89baa26a34a85f6e9c13e0c616d4b6": "h_n a trois axes : couches, exemples, état", "9238ca07937b1e9bd7d2d21d89fa7060d0f9b52e18a069a5055c4e8fa072e078": "h_n ne contient que le dernier pas de temps : c'est output qui les a tous"}}
📝 Ex 25.R2a : réponse enregistrée (str).
WB_ANSWER {"alt_coarse": ["26df1ba2f386c1c0a517184a4e56ee01644073e68b61096ceec45f47400d78bb"], "decimals": 1, "hash": "23ef5caf6f25970abe903798887f96c91d9e7ba9912de620e9aaf07a9b5bf2ce", "hash_coarse": "6ef8edc498d1b07fd2955e7f57451fb10b325fd8876e6753e0f5daa5703729e5", "id": "25.R2b", "kind": "float", "magnitude": 1, "mistakes": {"371617fc24df06f5307b0593b6583e2075683074aac29b6928722a208b69098b": "compte toutes les valeurs de l

**25.R3 — La divergence KL, ce que mesure le terme de régularisation du VAE**

In [17]:
wb.record("25.R3a", kl_bits([0.5, 0.5], [0.25, 0.75]), decimals=2, mistakes={"en bits, le logarithme est en base 2": 0.14,
          "KL(P‖Q) moyenne log(P/Q) sous P, pas sous Q": 0.19})
wb.record("25.R3b", False, mistakes={"calcule KL(Q‖P) avec les lois de a) et compare": True})
wb.record("25.R3c", True, mistakes={"chaque terme vaut p log(p / p) : que vaut log 1 ?": False})
wb.record("25.R3d", True, mistakes={"relis l'inégalité de Gibbs (ch. 6)": False})
wb.record("25.R3e", kl_bits([0.5, 0.5], [0.25, 0.75]) * math.log(2), decimals=2, mistakes={"pour passer des bits aux nats, on multiplie par ln 2": 0.30,
          "c'est la valeur en bits": 0.21})

WB_ANSWER {"decimals": 2, "hash": "7dc7ef26d515720638453428b2fa494be0c1611d0a21ebc785ce24de2e9e5abd", "hash_coarse": "4425fdddbe531ba5e8d22f32b6c7d76461f90e0d61a6967c0f2704d0d89551d2", "id": "25.R3a", "kind": "float", "magnitude": -1, "mistakes": {"16a45918df434598216ec1ded0bcaec2c96b59a8731c6f636512bcd24668644d": "KL(P‖Q) moyenne log(P/Q) sous P, pas sous Q", "c3c240f61038bd423b2a566d2ff6f8b8d2eb9445c72830d8375473209a1fc2f8": "en bits, le logarithme est en base 2"}, "sign": 1}
📝 Ex 25.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "a14b74f245bad31963a38576f64d994fba5ce5fe95e6bd0e7827357ff5f9ef2e", "id": "25.R3b", "kind": "bool", "mistakes": {"c8c0416d16c7f72babca171ff4480b76cf9c44907b30281f0e5cffc2e8e74c6c": "calcule KL(Q‖P) avec les lois de a) et compare"}}
📝 Ex 25.R3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "bc1140fff4f412fdf9086e410169a4548b0f5052ec5525e9a307af7c305d473b", "id": "25.R3c", "kind": "bool", "mistakes": {"8209896407903cdd1de52f0bc89fd795e

**Ex 25.1 — Compter les paramètres d'un autoencodeur à goulot**

In [18]:
wb.record("25.1a", dense(784, 20), mistakes={"un biais par neurone, en plus des poids": 784 * 20,
          "relis la question : l'encodeur a un biais par latent": dense(20, 784)})
wb.record("25.1b", dense(20, 784), mistakes={"relis la question : on demande le décodeur": dense(784, 20),
          "sans oublier un biais par pixel de sortie": 784 * 20})
wb.record("25.1c", dense(784, 20) + dense(20, 784), mistakes={"les biais comptent": 2 * 784 * 20})
wb.record("25.1d", P_ENC_DEEP, mistakes={"les biais comptent": P_ENC_DEEP - (512 + 256 + 128 + 20)})
wb.record("25.1e", P_ENC_DEEP + P_DEC_DEEP, mistakes={"le décodeur n'a pas les mêmes biais que l'encodeur : compte-le à part": 2 * P_ENC_DEEP,
          "les biais comptent": P_ENC_DEEP + P_DEC_DEEP - (512 + 256 + 128 + 20) - (128 + 256 + 512 + 784)})
wb.record("25.1f", 784 / 20, decimals=1, mistakes={"une image MNIST a 28 × 28 pixels": 28 / 20})
wb.record("25.1g", dense(10_000, 20) + dense(20, 10_000), mistakes={"les biais comptent": 400_000,
          "le décodeur a un biais par pixel de sortie": 400_040})

WB_ANSWER {"hash": "16c96b893b56332f57958a855e056a390c05a47f79449a8d537c4c594b48dc97", "id": "25.1a", "kind": "int", "magnitude": 4, "mistakes": {"058b3e75a071cf73d4addce4e8544cee533333474e7a8a864b73c68d2f16d027": "relis la question : l'encodeur a un biais par latent", "3ebe5ffe00eebccfc888446e22c65c81dcdab64646482e8c1de4ebbaa2c4a6b3": "un biais par neurone, en plus des poids"}, "sign": 1}
📝 Ex 25.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "96d1ecf150771ebb50ce7a5aebd0beb3535110e98ab6a3aa93e9a7e73f310505", "id": "25.1b", "kind": "int", "magnitude": 4, "mistakes": {"4e14021e2ee4d7723d324e70a9105b117998d215089f14f51f30a79d6ba6efd8": "relis la question : on demande le décodeur", "8fee4c5c010429fae7e725d99b28b27483341adf6026a8e67067b631adf04060": "sans oublier un biais par pixel de sortie"}, "sign": 1}
📝 Ex 25.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "2852446d7c8e38e0824fb7fb3b9cc61a1271a318fc4dac9798a7e68b3c8b331b", "id": "25.1c", "kind": "int", "magnitude": 4, "mistakes

**Ex 25.2 — Mélanger deux cercles : pixels contre paramètres**

In [19]:
wb.record("25.2a", "(4, 3)", mistakes={"t = 0,25 : un quart du chemin de A vers B, pas trois quarts": "(8, 5)",
          "t = 0,25, pas le milieu": "(6, 4)"})
wb.record("25.2b", MIX_25_2['r'], decimals=2, mistakes={"(1 − t) pour A, t pour B": 3.25,
          "t = 0,25, pas le milieu": 2.5})
wb.record("25.2c", MIX_25_2['g'], decimals=2, mistakes={"(1 − t) multiplie l'intensité de A": 0.55,
          "t = 0,25, pas le milieu de l'intensité": 0.7})
wb.record("25.2d", 0.75 * A_25_2['g'], decimals=2, mistakes={"ce pixel n'est que dans A : son poids est 1 − t": 0.25,
          "ce pixel est-il aussi dans B ?": 0.85})
wb.record("25.2e", 0.25 * B_25_2['g'], decimals=2, mistakes={"ce pixel n'est que dans B : son poids est t, sa valeur l'intensité de B": 0.3,
          "multiplie aussi par l'intensité de B": 0.25})
wb.record("25.2f", 2, mistakes={"dans un mélange de contenu, chaque disque garde sa place": 1})
wb.record("25.2g", math.pi * 2.5 ** 2, decimals=1, mistakes={"l'aire d'un disque est π r², pas 2 π r": 2 * math.pi * 2.5,
          "relis la question : l'aire du disque mélangé, pas la moyenne des aires": (math.pi + 16 * math.pi) / 2})
wb.record("25.2h", (math.pi * 1 ** 2 + math.pi * 4 ** 2) / 2, decimals=1, mistakes={"relis la question : la moyenne des aires des deux disques": math.pi * 2.5 ** 2,
          "π r² pour chaque disque, pas 2 π r": (2 * math.pi + 8 * math.pi) / 2})

WB_ANSWER {"hash": "ba47ad9d7a3360b93e26c01762d43541dfcb807b099c0798e6c688b0be6c72c5", "id": "25.2a", "kind": "str", "mistakes": {"2872953bcbf75bc0f6b994113325207dfe67df16014005effea4fa0b3dc02838": "t = 0,25 : un quart du chemin de A vers B, pas trois quarts", "726ee3086b57e2402c2a2a333e9c2fa87faf002f7b30d8587788d1d8ab56f314": "t = 0,25, pas le milieu"}}
📝 Ex 25.2a : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "7f0baa2ff70559b450f6af32ee867a496488a0a30a48f7846488d80df2c43cb2", "hash_coarse": "b78b512495cf24bbb5549d3db7e9b5a8eda9c949ca9498f7e97dea3a0732c0c4", "id": "25.2b", "kind": "float", "magnitude": 0, "mistakes": {"1ff3a4915eaf0ff9a35409f52ca3f6931f5ad16c97c594841f1494e03daf6231": "t = 0,25, pas le milieu", "23d3dab6ace507fbf57bfda7564b4b2e8dc528d7812d888189234e7c599f3cbe": "(1 − t) pour A, t pour B"}, "sign": 1}
📝 Ex 25.2b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["30f751c62128a6633d224e7d435a9ffe51cada73cd7397e5bbbfc2511d4f7547"], "d

**Ex 25.3 — Suivre les formes dans l'autoencodeur convolutif**

In [20]:
wb.record("25.3a", "(64, 16, 14, 14)", mistakes={"le pooling divise la hauteur et la largeur par deux": "(64, 16, 28, 28)",
          "PyTorch range les canaux avant la hauteur et la largeur": "(64, 14, 14, 16)",
          "l'axe des exemples reste en premier": "(16, 14, 14)"})
wb.record("25.3b", "(64, 8, 7, 7)", mistakes={"un pooling ne change pas le nombre de canaux : d'où viennent-ils ?": "(64, 7, 7, 7)",
          "les canaux sont ceux de la deuxième convolution": "(64, 16, 7, 7)",
          "les canaux avant la hauteur et la largeur": "(64, 7, 7, 8)"})
wb.record("25.3c", "(64, 3, 7, 7)", mistakes={"il n'y a pas de pooling après la troisième convolution": "(64, 3, 3, 3)",
          "les canaux avant la hauteur et la largeur": "(64, 7, 7, 3)"})
wb.record("25.3d", P_ENC_CONV, mistakes={"chaque filtre a aussi un biais": P_ENC_CONV - (16 + 8 + 3),
          "un noyau 3 × 3 a 9 poids par canal d'entrée": 16 * 1 * 3 + 16 + 8 * 16 * 3 + 8 + 3 * 8 * 3 + 3})
wb.record("25.3e", P_DEC_CONV, mistakes={"chaque convolution a un biais par filtre": P_DEC_CONV - (16 + 1),
          "les sur-échantillonnages n'ont pas de paramètres": P_DEC_CONV + 2})
wb.record("25.3f", P_ENC_CONV + P_DEC_CONV, mistakes={"les biais comptent": P_ENC_CONV + P_DEC_CONV - (16 + 8 + 3 + 16 + 1)})
wb.record("25.3g", 784 / (3 * 7 * 7), decimals=2, mistakes={"valeurs de l'image divisées par valeurs du code, pas l'inverse": 0.19,
          "le code a 3 canaux de 7 × 7": 16.0})
wb.record("25.3h", 32 * 7 * 7, mistakes={"multiplie aussi par les 32 canaux": 49,
          "il y a deux poolings": 32 * 14 * 14})

WB_ANSWER {"hash": "fa7670c099104a4f9135ac3b5d180bf0c0c6226b4a14bdfb19ecc52d2c2c364f", "id": "25.3a", "kind": "str", "mistakes": {"1dd43319e049e13209d8d77904158b2df168820fcde70f05436c57153671f461": "PyTorch range les canaux avant la hauteur et la largeur", "c4b7fc9d4b85387fd23159a043ae86fc35342ba84b6b2661d8dddc415c00c0df": "l'axe des exemples reste en premier", "e0a11c4bcf7e57090507e3cc2df600f3f5f294e8556d50bc6b867dddf0cef6f5": "le pooling divise la hauteur et la largeur par deux"}}
📝 Ex 25.3a : réponse enregistrée (str).
WB_ANSWER {"hash": "ba6d2af9ed0711fc748ee01db7f91edf6e78ed3b898906dd640e7d62142f1806", "id": "25.3b", "kind": "str", "mistakes": {"1850cd12405661106c2cdbe98fd4ebb15b416f2f1872ed90e814f047ab35bbe5": "les canaux avant la hauteur et la largeur", "ad79fd0274e91db82677a0b2a0ba8f118d1f48a2494e75a4f72ff7ed48edac45": "les canaux sont ceux de la deuxième convolution", "e2453f47b529fa1261232b812cad31abe6cd93f68fed15a2c81ce57d0032b5e5": "un pooling ne change pas le nombre de can

**Ex 25.4 — Retrouver 28×28 : padding et output_padding d'une convolution transposée**

In [21]:
wb.record("25.4a", tconv_out(7, 3, 2, 1), mistakes={"(H − 1) s, pas H s": 7 * 2 - 2 + 3,
          "le padding retire 2p, il n'ajoute pas": 6 * 2 + 2 + 3})
wb.record("25.4b", tconv_out(7, 3, 2, 1, 1), mistakes={"output_padding ajoute sa valeur à la sortie": tconv_out(7, 3, 2, 1)})
wb.record("25.4c", tconv_out(14, 3, 2, 1, 1), mistakes={"n'oublie pas output_padding": tconv_out(14, 3, 2, 1),
          "(H − 1) s, pas H s": 14 * 2 - 2 + 3 + 1})
wb.record("25.4d", tconv_out(7, 2, 2, 0), mistakes={"(H − 1) s, pas H s": 7 * 2 + 2})
wb.record("25.4e", tconv_out(7, 4, 2, 1), mistakes={"(H − 1) s − 2p + k : vérifie le premier terme et le padding": 7 * 2 - 2 + 4})
wb.record("25.4f", conv_out(7, 3, 2, 1), mistakes={"⌊(H + 2p − k)/s⌋ + 1 : vérifie le padding (des deux côtés) et le + 1": conv_out(7, 3, 2, 1) - 1})
wb.record("25.4g", tconv_out(conv_out(7, 3, 2, 1), 3, 2, 1), mistakes={"(H − 1) s − 2p + k, avec H la hauteur de f)": 4 * 2 - 2 + 3,
          "sans output_padding": tconv_out(conv_out(7, 3, 2, 1), 3, 2, 1, 1)})
wb.record("25.4h", 32 * 16 * 3 * 3 + 16, mistakes={"un biais par canal de sortie, en plus des poids": 32 * 16 * 3 * 3,
          "un biais par canal de sortie (16), pas d'entrée": 32 * 16 * 3 * 3 + 32})

WB_ANSWER {"hash": "c537a16e269d9ddadf5b5afd74935dc78936948c545e015b3e8cd494799c7f26", "id": "25.4a", "kind": "int", "magnitude": 1, "mistakes": {"7c097e45b7b5772ff9f1ea168f5d5eecbb2d046b5bb9712ade3ff320087104a1": "(H − 1) s, pas H s", "f45f0f48e8602a5bb4b3d52b36a2082215e127c7cf25fea6e69a2a2c7cf3f2fb": "le padding retire 2p, il n'ajoute pas"}, "sign": 1}
📝 Ex 25.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "d016ddaee1216bfb8745810c83c3bff78b2986c1f404d83572fb5324a15013ab", "id": "25.4b", "kind": "int", "magnitude": 1, "mistakes": {"0aa877c88deb953904130210fc122df280aac05808d3ca2cbcfe67e10ccb48c3": "output_padding ajoute sa valeur à la sortie"}, "sign": 1}
📝 Ex 25.4b : réponse enregistrée (int).
WB_ANSWER {"hash": "decde173b7d01589cb5cf685e8396e625249cb385edb55ea2ebcb291e42703e2", "id": "25.4c", "kind": "int", "magnitude": 1, "mistakes": {"3575f8e701b16cec166b86edafdbfcf7668158861f56b72fff4039cc99a98c14": "n'oublie pas output_padding", "8d0bcaf888cda5eadc1f8dc344a64d1c3f6b1820d347

**Ex 25.6 — Où tombent les tirages d'une gaussienne en grande dimension**

In [22]:
wb.record("25.6a", 20, mistakes={"E[z_j²] = 1 pour chacune des d coordonnées : additionne-les": 1})
wb.record("25.6b", math.sqrt(20), decimals=2, mistakes={"c'est E‖z‖² : on demande sa racine": 20.0})
wb.record("25.6c", 1 - math.exp(-1 / 2), decimals=3, mistakes={"1 − e^(−r²/2), pas e^(−r²/2)": math.exp(-1 / 2),
          "l'exposant est r²/2, pas r²": 1 - math.exp(-1)})
wb.record("25.6d", 1 - math.exp(-9 / 2), decimals=3, mistakes={"r = 3, donc r² = 9": 1 - math.exp(-3 / 2),
          "1 − e^(−r²/2), pas e^(−r²/2)": math.exp(-9 / 2)})
wb.record("25.6e", math.sqrt(2 * math.log(2)), decimals=2, mistakes={"on cherche r, pas r² : prends la racine": 2 * math.log(2),
          "ln 2 (logarithme népérien), pas log₁₀ 2": math.sqrt(2 * math.log10(2))})
wb.record("25.6f", 20 // 2, mistakes={"chaque coordonnée de m a pour variance 1/4 + 1/4": 20,
          "les deux tirages sont indépendants : leurs variances s'ajoutent avant la division par 4": 5})
wb.record("25.6g", math.sqrt(10), decimals=2, mistakes={"c'est E‖m‖² : on demande sa racine": 10.0})
wb.record("25.6h", math.sqrt(512), decimals=2, mistakes={"la norme typique est la racine de d": 512.0})

WB_ANSWER {"hash": "1b3731389095cdf8fac11e17cf93a0b952d507e85709a88678a7fd4697f3243f", "id": "25.6a", "kind": "int", "magnitude": 1, "mistakes": {"6d7a53b6919e36d989309e30ab5178ad2272fb8079e78c18cdbaeb155d2119d0": "E[z_j²] = 1 pour chacune des d coordonnées : additionne-les"}, "sign": 1}
📝 Ex 25.6a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "d3ba5512435ccfbfd0245ddb2966fe5f1f006fe47996c39f889b3451b0c7cb44", "hash_coarse": "2bec0f6de895e8b92a7b6b3dd516d42fe18f24d8c57bc4f111adcbfc657f9fe8", "id": "25.6b", "kind": "float", "magnitude": 0, "mistakes": {"928a0e00b1bb26740a2d0cdbd54bb7c8e6237b963cf98257e18ff4913b22de72": "c'est E‖z‖² : on demande sa racine"}, "sign": 1}
📝 Ex 25.6b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "397f5d1ac86b894f3bf16e3fa4f2d1269bd0efcb7585946d13592f66fadeec90", "hash_coarse": "6d63ffb87124ece36db6cdd99088ee498f819947e1d80c1c8040af55397ffff0", "id": "25.6c", "kind": "float", "magnitude": -1, "mistakes": {"3f

**Ex 25.7 — La divergence KL entre N(μ, σ²) et N(0, 1)**

In [23]:
wb.record("25.7a", gaussian_kl(2, 1), decimals=3, mistakes={"n'oublie pas le facteur ½": 4.0})
wb.record("25.7b", gaussian_kl(0, math.e), decimals=3, mistakes={"n'oublie pas le facteur ½": 2 * gaussian_kl(0, math.e),
          "σ² = e, et ln σ² = 1 : ne les échange pas": 0.5 * (1 - 1 - math.e)})
wb.record("25.7c", gaussian_kl(0, 0.01), decimals=3, mistakes={"σ = 0,1 donne σ² = 0,01 : la formule prend la variance": gaussian_kl(0, 0.1),
          "n'oublie pas le facteur ½": 2 * gaussian_kl(0, 0.01)})
wb.record("25.7d", gaussian_kl(1, 4), decimals=3, mistakes={"n'oublie pas le facteur ½": 2 * gaussian_kl(1, 4),
          "le logarithme est celui de σ² = 4, pas de σ": 0.5 * (1 + 4 - 1 - math.log(2))})
wb.record("25.7e", gaussian_kl(1, 1) + gaussian_kl(0, 1) + gaussian_kl(-1, 1), decimals=3, mistakes={"on additionne les KL des latents, on ne les moyenne pas": 1 / 3})
wb.record("25.7f", gaussian_kl(0.5, 2.5), decimals=3, mistakes={"n'oublie pas le facteur ½": 2 * gaussian_kl(0.5, 2.5),
          "μ² = 0,25, pas 0,5": 0.5 * (0.5 + 2.5 - 1 - math.log(2.5))})
wb.record("25.7g", 0.5 * (5 - 1), decimals=3, mistakes={"n'oublie pas le facteur ½": 4.0,
          "dérive ½(e^s − 1 − s), pas ½ e^s": 2.5})
wb.record("25.7h", gaussian_kl(1, 4) / math.log(2), decimals=3, mistakes={"pour passer des nats aux bits, on divise par ln 2": gaussian_kl(1, 4) * math.log(2),
          "c'est la valeur en nats": gaussian_kl(1, 4)})

WB_ANSWER {"decimals": 3, "hash": "f16f819128f0f700bc6f3ad147072a8a05653e10c33c3402f8f219c747ced935", "hash_coarse": "7d66d3ffa246a105ba98a13ee9d74c478f13c8f8f41518627cd3adb5832ae4a7", "id": "25.7a", "kind": "float", "magnitude": 0, "mistakes": {"b06b9de479a26b05bba7c44d6682f74819c45e050895fdbb9bbee38cf5380c55": "n'oublie pas le facteur ½"}, "sign": 1}
📝 Ex 25.7a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "d595bb053ac5f980863fef84a3dc14b475c4dbcba50ef58fd5f3ba7176565fec", "hash_coarse": "4c655ec55446ae24a0c74fdd9d0baf081aa313d005e4cc3296faf54a5a78f72e", "id": "25.7b", "kind": "float", "magnitude": -1, "mistakes": {"5bab33430a4e95f9455d673264c187dbcc7aa19e6c24d1c93bda399cd561d25d": "n'oublie pas le facteur ½", "ae3ef1773691a62b37bcc35e86280654e362d8678b7a78320893569d74e5d123": "σ² = e, et ln σ² = 1 : ne les échange pas"}, "sign": 1}
📝 Ex 25.7b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "03f0b57c891745b974b4fafa7b9

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu expliquer le rôle du goulot, et pourquoi un autoencodeur entraîné sur une seule image ne comprime rien ?
2. Sais-tu dire ce que l'autoencodeur linéaire a en commun avec l'ACP, et ce qu'il ne retrouve pas ?
3. Sais-tu écrire la reparamétrisation et la KL gaussienne, et expliquer pourquoi un VAE sait générer ?

**Pour aller plus loin** : l'introduction aux VAE de Kingma et Welling (2019), citée dans la fiche. La suite : le ch. 26, l'apprentissage par renforcement, puis les GAN au ch. 27.

*Les exercices de code 25.14 à 25.39 (parties A à C : les autoencodeurs denses, convolutifs et débruiteurs, puis le VAE, ta librairie `vae.py` et le défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 25` pour obtenir le notebook complet.*